# MATH GR5360 Final Project — Notebook 01

Segmented diagnostics notebook. This notebook documents where the trend-following property appears in the time series, with special emphasis on the professor-requested figures: a dense variance-ratio curve and selected push-response diagrams at the reference `tau` horizon.

In [ ]:
MARKET_SELECT = 'TY'   # 'TY' (primary) or 'BTC' (secondary)
QUICK_TEST = True

In [ ]:
from pathlib import Path
import sys
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

CWD = Path.cwd().resolve()
PROJECT_ROOT = CWD if (CWD / 'mafn_engine').exists() else CWD.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from mafn_engine import (
    COLUMBIA_CORE,
    COLUMBIA_DIVERGING,
    COLUMBIA_NAVY,
    COLUMBIA_RED,
    COLUMBIA_WARM,
    apply_columbia_theme,
    bars_to_time,
    get_market,
    load_ohlc,
    professor_reference_tau,
    prepare_analysis_frame,
    run_diagnostics,
    validate_ohlc,
)

apply_columbia_theme()
MARKET = get_market(MARKET_SELECT)
DATA_DIR = str(PROJECT_ROOT / 'data')
print(f"Market: {MARKET.ticker} - {MARKET.name} ({MARKET.exchange})")
print(f"PV=${MARKET.PV:,}  Slippage=${MARKET.slpg}  E0=${MARKET.E0:,.0f}")

In [ ]:
def ensure_diagnostics_state(force: bool = False) -> None:
    global MARKET, full_df, validation, analysis_df, return_summary
    global diagnostics_bundle, vr_price_df, vr_logret_df, pr_summary_df
    global trend_profile, representative_pr, diagnostic_scales_df
    global professor_bundle, vr_curve_df, short_pr, reference_pr
    global reference_tau, reference_scale, short_tau, short_scale
    global _analysis_market, _diagnostics_market

    MARKET = get_market(MARKET_SELECT)

    if force or globals().get('_analysis_market') != MARKET_SELECT or 'analysis_df' not in globals():
        full_df = load_ohlc(DATA_DIR, MARKET_SELECT, fallback_synthetic=False)
        validation = validate_ohlc(full_df)
        analysis_df = prepare_analysis_frame(full_df, MARKET_SELECT)
        logret = np.log(full_df['Close'] / full_df['Close'].shift(1)).dropna()
        return_summary = {
            'mean': float(logret.mean()),
            'std': float(logret.std()),
            'skew': float(logret.skew()),
            'ex_kurtosis': float(logret.kurt()),
        }
        _analysis_market = MARKET_SELECT

    if force or globals().get('_diagnostics_market') != MARKET_SELECT or 'diagnostics_bundle' not in globals():
        diagnostics_bundle = run_diagnostics(analysis_df, MARKET_SELECT)
        vr_price_df = diagnostics_bundle['vr_price_df']
        vr_logret_df = diagnostics_bundle['vr_logret_df']
        pr_summary_df = diagnostics_bundle['pr_summary_df']
        trend_profile = diagnostics_bundle['trend_profile']
        representative_pr = diagnostics_bundle['representative_pr']
        diagnostic_scales_df = vr_price_df[['time_scale', 'k', 'VR', 'Z2', 'significant']].copy()
        diagnostic_scales_df = diagnostic_scales_df.rename(columns={'significant': 'robust_sig'})

        professor_bundle = diagnostics_bundle['professor_bundle']
        vr_curve_df = professor_bundle['vr_curve_df']
        short_pr = professor_bundle['short_pr']
        reference_pr = professor_bundle['reference_pr']
        reference_tau = int(professor_bundle['reference_tau'])
        reference_scale = professor_bundle['reference_scale']
        short_tau = int(professor_bundle['short_tau'])
        short_scale = professor_bundle['short_scale']
        _diagnostics_market = MARKET_SELECT


ensure_diagnostics_state(force=True)

print()
print('DATA VALIDATION')
print('-' * 72)
print(f"Full bars:        {validation['n_bars']:,}")
print(f"Session bars:     {len(analysis_df):,}")
print(f"Date range:       {validation['start']} -> {validation['end']}")
print(f"Years covered:    {validation['years']:.1f}")
print(f"Validation OK:    {validation['is_valid']}")
if validation['issues']:
    print(f"Issues:           {validation['issues']}")

print()
print('LOG-RETURN SUMMARY (appendix only)')
print('-' * 72)
print(
    f"mean={return_summary['mean']:+.3e}  std={return_summary['std']:.5f}  "
    f"skew={return_summary['skew']:+.3f}  ex-kurt={return_summary['ex_kurtosis']:+.3f}"
)

In [ ]:
ensure_diagnostics_state()

month_sessions = reference_tau / MARKET.bars_per_session
trend_summary_df = pd.DataFrame(
    [
        {
            'Assignment Strategy': 'Trend Following',
            'Reference tau (bars)': reference_tau,
            'Reference scale': reference_scale,
            'Approx sessions': month_sessions,
            'TF Bias': trend_profile['tf_speed_bias'],
            'Short VR Mean': trend_profile['vr_short_mean'],
            'Long VR Mean': trend_profile['vr_long_mean'],
            'Long - Short VR': trend_profile['vr_long_minus_short'],
            'Peak VR Scale': trend_profile['peak_vr_scale'],
            'Peak PR Scale': trend_profile['peak_pr_scale'],
        }
    ]
)

print()
print('PROFESSOR GUIDANCE SUMMARY')
print('-' * 72)
if MARKET_SELECT == 'TY':
    print(
        'TY should be described as short-horizon mean-reverting / weak, but gradually more trend-following as q and tau lengthen. '
        'The key point is not that VR must exceed 1 immediately; it is that the variance ratio turns upward as the horizon stretches '
        f'toward tau≈{reference_tau} bars, which is about {month_sessions:.1f} liquid sessions, roughly one trading month.'
    )
    print(
        'Economic interpretation to mention: rates consensus and macro repricing take time, so the trend-following property emerges '
        'with delay rather than instantly on daily horizons.'
    )
elif MARKET_SELECT == 'BTC':
    print(
        f'BTC is expected to show trend-following much more clearly, with a strong multi-day reference horizon around tau≈{reference_tau} bars.'
    )
print()
print(f"Assignment strategy family: TF (Channel WithDDControl)")
print(f"Reference tau:             {reference_tau} bars = {reference_scale}")
print(f"Short comparison tau:      {short_tau} bars = {short_scale}")
print(f"Recommended TF bias:       {trend_profile['tf_speed_bias'].upper()}")
print(f"Peak VR scale:             {trend_profile['peak_vr_scale']}  (VR={trend_profile['peak_vr']:.3f})")
if np.isfinite(trend_profile['peak_pr_rho']):
    print(f"Peak PR scale:             {trend_profile['peak_pr_scale']}  (rho={trend_profile['peak_pr_rho']:+.3f})")
trend_summary_df

In [ ]:
ensure_diagnostics_state()

fig, axes = plt.subplots(1, 2, figsize=(15, 5.4))

axes[0].plot(vr_curve_df['q'], vr_curve_df['VR'], color=COLUMBIA_NAVY, linewidth=2)
axes[0].axhline(1.0, color=COLUMBIA_RED, linestyle='--', linewidth=1.2)
axes[0].axvline(reference_tau, color=COLUMBIA_CORE, linestyle='--', linewidth=1.5)
axes[0].set_title('Dense Variance Ratio Curve')
axes[0].set_xlabel('q (bars)')
axes[0].set_ylabel('Variance Ratio')
axes[0].annotate(
    f"reference tau={reference_tau}",
    xy=(reference_tau, float(vr_curve_df.loc[vr_curve_df['q'] == reference_tau, 'VR'].iloc[0]) if (vr_curve_df['q'] == reference_tau).any() else 1.0),
    xytext=(reference_tau * 1.03, vr_curve_df['VR'].max()),
    color=COLUMBIA_NAVY,
    arrowprops={'arrowstyle': '->', 'color': COLUMBIA_NAVY},
)

axes[1].plot(vr_curve_df['q'], vr_curve_df['VR'] - 1.0, color=COLUMBIA_WARM, linewidth=2)
axes[1].axhline(0.0, color=COLUMBIA_RED, linestyle='--', linewidth=1.2)
axes[1].axvline(reference_tau, color=COLUMBIA_CORE, linestyle='--', linewidth=1.5)
axes[1].set_title('Dense Variance Ratio Minus One')
axes[1].set_xlabel('q (bars)')
axes[1].set_ylabel('VR(q) - 1')

plt.suptitle(f"{MARKET.ticker} - Professor-Requested Variance-Ratio Figure", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
ensure_diagnostics_state()

fig, axes = plt.subplots(1, 2, figsize=(15, 5.4))

for ax, diag, title in [
    (axes[0], short_pr, f"Push-Response at short tau={short_tau}"),
    (axes[1], reference_pr, f"Push-Response at reference tau={reference_tau}"),
]:
    if diag is not None:
        ax.errorbar(
            diag['bin_centre'],
            diag['bin_mean'],
            yerr=1.96 * diag['bin_se'],
            color=COLUMBIA_NAVY,
            marker='o',
            linewidth=1.4,
        )
        ax.axhline(0.0, color=COLUMBIA_RED, linestyle='--', linewidth=1)
        ax.axvline(0.0, color=COLUMBIA_CORE, linestyle='--', linewidth=1)
        ax.set_title(title)
        ax.set_xlabel('push')
        ax.set_ylabel('average response')
    else:
        ax.text(0.5, 0.5, 'Not enough observations for this tau', ha='center', va='center')
        ax.set_axis_off()

plt.suptitle(f"{MARKET.ticker} - Professor-Requested Push-Response Figures", y=1.02)
plt.tight_layout()
plt.show()

diagnostic_scales_df

In [ ]:
ensure_diagnostics_state()

print()
print('FINAL DIAGNOSTIC SUMMARY')
print('-' * 72)
print('Assignment strategy family: TF (Channel WithDDControl)')
if MARKET_SELECT == 'TY':
    print(
        f"Short horizons look mean-reverting / weak, but the dense VR curve bends upward and the push-response turns more TF-consistent around tau≈{reference_tau} bars."
    )
    print(
        'This is the figure set to present in the project: it shows why the treasury trend-following horizon should be chosen around one month rather than around one day.'
    )
elif MARKET_SELECT == 'BTC':
    print(
        f"BTC shows the TF property much earlier and more clearly, so a shorter multi-day TF horizon is reasonable; reference tau≈{reference_tau} bars."
    )
print(trend_profile['narrative'])